# Notebook 10: look at our test data on Hugging Face
Lesson: [10 Hugging Face](../lessons/10-hugging-face.md). **No GPU needed** (Colab: Runtime → change runtime type → CPU is fine).

We only **read** problems and count them. We never run any code from the datasets or from a model here.

## 1. Install the libraries
**Problem:** the notebook computer may not have the Hugging Face libraries. **Why:** we need them to download data.
**In:** nothing. **Out:** installed `datasets` and `huggingface_hub`. **Why this way:** `pip` is the standard free installer.

In [ ]:
!pip install -q datasets huggingface_hub

## 2. HumanEval+: load and count
**Problem:** check the test set size ourselves. **Why:** PLAN.md says 164; "checked" beats "assumed".
**In:** the dataset id `evalplus/humanevalplus`. **Out:** number of problems and the column names. **Why this way:** `load_dataset` downloads and caches it in one line.

In [ ]:
from datasets import load_dataset

humaneval = load_dataset("evalplus/humanevalplus", split="test")
print("HumanEval+ problems:", len(humaneval))      # we expect 164
print("columns:", humaneval.column_names)

## 3. Read one problem like the model will see it
**Problem:** what does a test problem look like? **Why:** so you know what the model is asked.
**In:** the first problem. **Out:** its id, the prompt, and the function name the tests call. **Why this way:** printing one row is the fastest way to understand a dataset.

Note: we don't print the `test` column's hidden checks in detail, and we never run them here. Grading happens later, in a sandbox (lesson 14).

In [ ]:
p = humaneval[0]
print(p["task_id"])
print(p["prompt"])
print("function the tests will call:", p["entry_point"])

## 4. MBPP+: load and count
**Problem:** check the second test set size. **Why:** PLAN.md says 378.
**In:** `evalplus/mbppplus`. **Out:** number of problems. **Why this way:** same one-line loader.

In [ ]:
mbpp = load_dataset("evalplus/mbppplus", split="test")
print("MBPP+ problems:", len(mbpp))                # we expect 378
print("columns:", mbpp.column_names)

## 5. LiveCodeBench: count the fresh problems by date and difficulty
**Problem:** which LiveCodeBench problems are after Gemma's cutoff (January 2025)? **Why:** lesson 06.
**In:** the file `test6.jsonl` (134 MB, the newest part). **Out:** counts per month and difficulty. **Why this way:** this dataset uses a loading script that newer libraries may refuse, so we download the raw file directly and read it line by line.

We counted this ourselves on 2026-09-17: from February 2025 on, 31 easy + 39 medium + 61 hard. Check that you get the same.

In [ ]:
import json, collections
from huggingface_hub import hf_hub_download

path = hf_hub_download("livecodebench/code_generation_lite", "test6.jsonl", repo_type="dataset")

by_month = collections.Counter()
fresh = collections.Counter()
with open(path) as f:
    for line in f:
        d = json.loads(line)
        month = d["contest_date"][:7]
        by_month[month] += 1
        if month >= "2025-02":          # after the cutoff month = fresh
            fresh[d["difficulty"]] += 1

for month in sorted(by_month):
    print(month, by_month[month])
print("fresh (from 2025-02):", dict(fresh))